# 02A — Correcting AR cubes into analysis-ready cubes

This notebook is the **correction** step, and nothing else. It takes the raw cubes
`01A_download_data.ipynb` produced and writes corrected ones.

**It builds no regions.** Umbra, penumbra, hot spot and the quiet-sun mask are all built in
`03A_data_analisis.ipynb` from the cubes written here.

That division is the point. The two corrections below re-read every per-frame FITS header,
which costs ~90 s per region, and none of that work depends on where an umbra boundary is
drawn. Keeping the regions out of here means a question that comes up while looking at the
plots — is the hot spot on the right polarity? is the tracker holding the spot? is the umbra
threshold too tight? — costs one 03A run instead of a full re-correction of four regions.

For every region (`data/raw/NOAA_<noaa>_<date>/region_01_*_cube.fits`) it:

1. Divides the continuum by its **limb-darkening** factor, the fifth Castellanos Durán &
   Kleint (2020) correction (`src/limb_darkening.py`).
2. Calibrates the dopplergram with the four Castellanos Durán et al. (2021) zero-point
   corrections, giving absolute m/s (`src/doppler_calibration.py`).
3. Places all three cubes on one **uniform time grid**, NaN-filling any slot a series has
   no frame for.
4. Optionally crops to the window where every frame has data, for regions whose download
   box changed mid-window.
5. Subtracts a fitted **plane** from the magnetogram, removing its instrumental offset and
   the gradient across the box.
6. Writes everything to `data/processed/NOAA_<noaa>_<date>/`.

Steps 1 and 2 run *before* step 3 and are applied per frame, because both depend on where
the tracked box is pointing at that instant.

## Outputs

| File | Contents |
| --- | --- |
| `region_01_continuum_cube.fits` | limb-darkening corrected, still DN/s |
| `region_01_magnetogram_corrected_cube.fits` | quiet-sun plane removed |
| `region_01_dopplergram_calibrated_cube.fits` | absolute LOS velocity, m/s |
| `region_01_frames.fits` | per-frame correction diagnostics — `PRESENT_*`, `I_QS`, `C_MEAN`, `V_SDO`/`V_LSF`/`V_CLV`/`V_GRAVITY`, `MAG_PLANE_GRAD` |

The three cubes carry the `TIMESTAMPS` extension and `HISTORY` cards recording which
corrections were applied, and open directly in **DS9** as cubes with a frame slider. There
is deliberately **no mask file** — 03A rebuilds the regions from these cubes on every run,
so a saved mask could only ever go stale against the thresholds currently set there. 03A
exports its own mask cube for DS9 when you want one.

## The one mask that is still needed here

The magnetogram plane has to be fitted over quiet-sun pixels, so step 5 needs to know which
pixels those are. It uses its own pinned `PLANE_QSUN_FRAC` / `PLANE_QSUN_PERCENTILE` cut
rather than 03A's segmentation thresholds. Two reasons: the fit only needs *most* of the
quiet sun and is insensitive to the exact cut, and pinning it means the magnetogram cube on
disk does not silently change meaning when a threshold is retuned downstream.

## The time axis

The three series are joined on a grid that is **uniform by construction**, covering the
union of their timestamps, rather than on the timestamps they happen to share. A frame a
series is missing becomes a **NaN frame in the right slot**. 03A recovers which slots those
are directly from the cubes — an all-NaN frame is unambiguous — so nothing has to stay in
sync.

The alternative — intersecting the timestamps — was what this notebook used to do, and it
fails in two ways at once. It throws away good frames from the other two series, and it
leaves *holes in the time axis*: NOAA 11536's dopplergram is missing 2012-08-01 09:48 and
2012-08-02 21:48, which put two 1440 s jumps into an otherwise 720 s series. Every FFT in
`src/sunspot_analysis.py` assumes one cadence, so all of them were quietly wrong.

**Caveats these cubes carry**, and that you should not read past when interpreting anything
downstream:
- Magnetogram values are **signed**, and no `cos θ` correction is applied — `B_los` changes
  purely geometrically as the region rotates, so part of any magnetogram trend is
  projection, not field evolution.
- The continuum stays in **DN/s**: Eq. 3's DN→cgs factor is deliberately not applied, since
  everything measured downstream is a ratio of intensities.

**Requires cubes built by the current `make_cube`** — older ones have no `TIMESTAMPS`
extension and `read_cube` will refuse them rather than fall back to re-globbing the frame
directory, which cannot detect a missing mid-window frame. The per-frame files in
`data/raw/<region>/region_01/` must also still be there: both the limb-darkening and the
Doppler corrections read the per-frame headers back from them.

In [1]:
import sys
import pathlib

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np

from src.utilities import (read_cube, write_cube, regular_time_grid, reindex_on_grid,
                           reindex_series_on_grid, crop_to_common_window)
from src.doppler_calibration import calibrate_cube
from src.limb_darkening import limb_darkening_cube, U_LAMBDA, V_LAMBDA

DATA_DIR      = pathlib.Path('../data/raw')
PROCESSED_DIR = pathlib.Path('../data/processed')

# --- Dopplergram zero point -------------------------------------------------
# CALIBRATE_DOPPLER applies the four physical corrections of Castellanos Durán et al.
# (2021) Sect. 2 — observatory velocity, large-scale flows, convective blueshift CLV,
# gravitational redshift — giving velocities on an absolute m/s scale.
#
# RESIDUAL_PLANE_FIT additionally fits and subtracts a plane over the quiet-sun pixels.
# That is the *empirical* correction this notebook used to rely on, and it is off by
# default on purpose: it flattens whatever gradient is left, which also throws away the
# absolute scale the physical corrections just established. Turn it on only to inspect
# what the physical model failed to remove, and don't read absolute velocities off the
# result when you do.
CALIBRATE_DOPPLER  = True
RESIDUAL_PLANE_FIT = False

# --- Continuum limb darkening -----------------------------------------------
# The fifth cleaning step, Castellanos Durán & Kleint (2020) Eqs. 1-2:
#
#     I_corrected = I_observed / C(mu),   C = 1 - u - v + u mu + v mu^2
#
# with u = 0.836, v = -0.204 at the 6173.3 A line HMI observes (src/limb_darkening.py).
# C is bounded in [0.368, 1], so this only ever brightens and can never flip a sign.
#
# **Continuum only.** Limb darkening is an intensity effect; the magnetogram and
# dopplergram are not divided by C — their zero-point corrections are the four above.
#
# MU_METHOD picks how cos(Theta) is obtained: 'geometry' goes through sunpy's transforms
# (finite observer distance), 'paper' is Eq. 2 literally (observer at infinity, ~3x
# faster, ~0.1% intensity difference near the limb). See src/limb_darkening.mu_from_map.
#
# Eq. 3's DN -> cgs factor (DN_TO_CGS = 52.5) is deliberately NOT applied: every quantity
# downstream is a *ratio* of intensities, so a global scale cancels out of all of them
# while making every number harder to compare against the raw frames and against DS9.
CORRECT_LIMB_DARKENING = True
MU_METHOD              = 'geometry'

# --- Quiet-sun pixels for the magnetogram plane fit -------------------------
# **These are not analysis knobs.** They exist only to decide which pixels the magnetogram
# plane is fitted over: everything brighter than PLANE_QSUN_FRAC x I_qs, where I_qs is that
# frame's own PLANE_QSUN_PERCENTILE of the continuum.
#
# The umbra/penumbra/hot-spot thresholds that used to live here have moved to
# 03A_data_analisis.ipynb, where the regions are now built. Do not re-add them: the point
# of the split is that changing a region definition costs one 03A run instead of a full
# re-correction here.
#
# The plane fit only needs "most of the quiet sun" and is insensitive to the exact cut, so
# it is safe to pin these rather than tie them to whatever 03A currently uses. Leaving them
# fixed is also what keeps the written magnetogram reproducible: the cube on disk does not
# silently change meaning when a threshold is retuned downstream.
PLANE_QSUN_FRAC       = 0.90   # pixels with I >= 0.90 I_qs are quiet sun for the fit
PLANE_QSUN_PERCENTILE = 80     # percentile of finite continuum pixels used as I_qs

# --- Cropping to the common data window -------------------------------------
# A tracked cutout is supposed to keep a constant pixel size for the whole window, so
# normally there is nothing to crop and this stays off. NOAA 11117 is the exception on
# two counts at once:
#
#   - its box shrinks partway through the window, and make_cube center-crops/NaN-pads the
#     smaller frames onto the majority 433x433 grid — that NaN border is never selected by
#     any threshold, so the mask areas step down for exactly as long as the smaller box
#     lasts, which looks like the spot shrinking;
#   - its magnetogram was downloaded under a different box again (402x402), so the three
#     cubes cannot even be indexed against each other.
#
# With the flag on, crop_to_common_window trims all three cubes to the largest rectangle
# in which *every* frame of *every* series has data. The saved cubes' CRPIX is shifted to
# match, so DS9 still puts them on the right sky.
#
# This is a repair for a bad download, not part of the pipeline: the real fix is to
# re-download 11117 with one consistent box. It belongs here rather than in 03A because it
# changes the pixel grid the cubes are written on.
crop_override = {
    11117: True,
}

## Discover regions

Any `NOAA_*` directory with a `region_01_continuum_cube.fits` — this naturally skips the pre-HMI ARs (11039, 11041), which have no cube files at all.

In [2]:
regions = []
for region_dir in sorted(DATA_DIR.glob('NOAA_*')):
    cont_cube = region_dir / 'region_01_continuum_cube.fits'
    if cont_cube.exists():
        regions.append(region_dir)

print(f'{len(regions)} region(s) found:')
for r in regions:
    print(' ', r.name)

4 region(s) found:
  NOAA_11106_2010-09-16
  NOAA_11117_2010-10-27
  NOAA_11363_2011-12-06
  NOAA_11536_2012-07-31


## Per-region correction

`process_region` loads the three cubes, corrects them, and places them on **one uniform
time grid** built from the union of their timestamps. Individual JSOC files do fail, and
when a frame is missing mid-window — which the download logs show happens — the slot stays
in the axis and its frame is NaN, rather than the frame being dropped and every later frame
sliding one slot out of step.

It then:

- divides the continuum by `C(mu)` through `limb_darkening_cube`, which reads the per-frame
  headers back from the frame directory (the box is tracked, so `mu` changes every frame);
- calibrates the dopplergram through `calibrate_cube`, which reads those same headers back
  for the same reason (`OBS_V*` changes every frame);
- subtracts a fitted **plane** from the magnetogram rather than a scalar mean, which takes
  out the gradient across the box and not just the uniform offset.

Both corrections run *before* the grid join, on the frames that exist; their per-frame
diagnostics (`c_means`, `doppler_terms`) are moved onto the grid alongside the cubes.

And then it stops. No masks, no hot spot, no time series, no plots — `build_regions` in
`src/sunspot_analysis.py`, driven from 03A, does all of that from the written cubes.

A gap slot flows through every step as an all-NaN frame, so its `I_qs` and plane
coefficients come back NaN rather than raising. `present` is carried through so 03A can tell
a gap apart from a frame in which nothing was detected.

In [3]:
def load_aligned(region_dir, calibrate=CALIBRATE_DOPPLER, limb_darken=CORRECT_LIMB_DARKENING,
                 crop_to_data=False, v_sdo_by_time=None):
    """Load the three cubes, correct them, and place them on one uniform time grid.

    The three series are downloaded independently and individual files do fail, so their
    frame counts differ. Two joins were considered and only one of them is safe:

    - *Intersection* of the timestamps (what this notebook used to do). It keeps only
      frames all three series have, which throws away good frames from the others and,
      worse, leaves holes in the time axis. NOAA 11536's dopplergram is missing
      2012-08-01 09:48 and 2012-08-02 21:48, so the intersection has two 1440 s jumps in
      an otherwise 720 s series — and every FFT in src/sunspot_analysis.py assumes a
      single cadence, so all of them come out quietly wrong.
    - A *uniform grid* covering the union of the timestamps, which is what happens here.
      Every series gets one frame per grid slot; where a series has no frame the slot is
      NaN. The time axis is then uniform by construction rather than inherited from
      whatever happened to download, and a missing mid-window frame stays visible as a
      gap instead of shifting every later frame out of step.

    Both corrections are applied *before* the join and per frame, because both depend on
    where the tracked box is pointing: the OBS_V* keywords change every frame, and mu at
    the box centre runs 0.78 -> 0.87 -> 0.85 across NOAA 11536's window. calibrate_cube
    and limb_darkening_cube therefore read the per-frame headers back from the region's
    frame directory (make_cube keeps only the reference frame's header).

    Parameters
    ----------
    crop_to_data : bool
        Trim all three cubes to the window in which every frame has data — see
        `crop_to_common_window` and the `crop_override` dict below. Off by default because
        it is a repair for a bad download, not part of the pipeline: on a healthy region it
        finds nothing to trim and only costs a pass over the cubes.

    Returns
    -------
    dict with keys
        cubes         {'cont', 'mag', 'dop'}, each (n_slots, ny, nx) float32
        grid          list of datetimes, one per slot, evenly spaced
        cadence_s     the grid spacing
        present       {'cont', 'mag', 'dop'} bool arrays — True where the series has data
        doppler_terms per-term spatial means on the grid, or None
        c_means       per-frame spatial mean of the limb-darkening factor C, or None
        crop_offset   (row0, col0) of the crop in the continuum's original pixels, or None
    """
    cubes, times = {}, {}
    for name, fname in [('cont', 'continuum'), ('mag', 'magnetogram'), ('dop', 'dopplergram')]:
        data, ts = read_cube(region_dir / f'region_01_{fname}_cube.fits')
        cubes[name], times[name] = data.astype(np.float32), ts

    c_means = None
    if limb_darken:
        corrected, cont_times, c_means = limb_darkening_cube(
            region_dir, method=MU_METHOD, u_lambda=U_LAMBDA, v_lambda=V_LAMBDA)
        if cont_times != times['cont']:
            raise ValueError('limb_darkening_cube returned different timestamps than the cube')
        cubes['cont'] = corrected

    term_means = None
    if calibrate:
        corrected, dop_times, term_means = calibrate_cube(region_dir, v_sdo_by_time=v_sdo_by_time)
        if dop_times != times['dop']:
            raise ValueError('calibrate_cube returned different timestamps than the cube')
        cubes['dop'] = corrected

    grid, cadence_s = regular_time_grid([times['cont'], times['mag'], times['dop']])

    present = {}
    for name in cubes:
        cubes[name], present[name] = reindex_on_grid(cubes[name], times[name], grid, cadence_s)

    # The per-frame diagnostics have to move onto the same grid as the cubes they describe,
    # or they end up plotted against the wrong times.
    if term_means is not None:
        term_means = {k: reindex_series_on_grid(v, times['dop'], grid, cadence_s)
                      for k, v in term_means.items()}
    if c_means is not None:
        c_means = reindex_series_on_grid(c_means, times['cont'], grid, cadence_s)

    gaps = {k: np.flatnonzero(~v) for k, v in present.items()}
    n_gaps = {k: len(v) for k, v in gaps.items()}
    print(f'  {len(grid)} slots on a uniform {cadence_s:.0f} s grid, '
          f'{grid[0]:%Y-%m-%d %H:%M} .. {grid[-1]:%Y-%m-%d %H:%M}')
    if any(n_gaps.values()):
        print(f'  NaN frames (no data): {n_gaps}')
        for name, idx in gaps.items():
            for i in idx[:5]:
                print(f'      {name}: slot {i} = {grid[i]:%Y-%m-%d %H:%M}')
            if len(idx) > 5:
                print(f'      {name}: ... and {len(idx) - 5} more')
    else:
        print('  no gaps — all three series cover every slot')

    crop_offset = None
    if crop_to_data:
        cubes, offsets = crop_to_common_window(cubes, present=present)
        crop_offset = offsets['cont']
    else:
        # Everything downstream indexes the magnetogram and dopplergram with masks built
        # from the continuum, so a shape mismatch has to stop here with an explanation
        # rather than as a broadcasting error 60 lines later.
        shapes = {name: cube.shape[1:] for name, cube in cubes.items()}
        if len(set(shapes.values())) > 1:
            raise ValueError(
                f'{region_dir.name}: the three series are on different pixel grids '
                f'({shapes}) — they were downloaded under different boxes. Either '
                f're-download the region with one consistent box, or set this region to '
                f'True in crop_override to trim all three to their common data window.')

    return dict(cubes=cubes, grid=grid, cadence_s=cadence_s, present=present,
                doppler_terms=term_means, c_means=c_means, crop_offset=crop_offset)


def remove_quiet_sun_plane(frame, qsun_mask, xn, yn):
    """Fit a plane to the quiet-sun pixels and subtract it from the whole frame.

    Subtracting a scalar quiet-sun mean only removes the spatially uniform term — for the
    dopplergram that is mostly the SDO orbital velocity. What survives is the line-of-sight
    solar-rotation gradient across the box, and because the umbra sits off to one side of
    the box its mean picks up a residual that drifts as the region rotates. Removing a
    plane instead takes out that gradient to first order.

    This is the *empirical* alternative to the physical corrections in
    src/doppler_calibration.py. It always applies to the magnetogram (which those
    corrections don't address) but only to the dopplergram when RESIDUAL_PLANE_FIT is set.

    A NaN (gap) frame has no finite quiet-sun pixels, so it returns unchanged with NaN
    coefficients rather than raising.

    Returns (corrected_frame, coefficients) with coefficients = (offset, d/dx, d/dy) in
    the normalized coordinates xn, yn.
    """
    valid = qsun_mask & np.isfinite(frame)
    if valid.sum() < 3:
        return frame, np.array([np.nan, np.nan, np.nan])
    design = np.column_stack([np.ones(valid.sum()), xn[valid], yn[valid]])
    coef, *_ = np.linalg.lstsq(design, frame[valid].astype(np.float64), rcond=None)
    plane = coef[0] + coef[1] * xn + coef[2] * yn
    return (frame - plane).astype(np.float32), coef


def process_region(region_dir, crop_to_data=False, v_sdo_by_time=None):
    """Correct one region's three cubes. Builds no regions and computes no metrics.

    This is where the notebook stops. Umbra, penumbra, hot spot and the quiet-sun mask are
    all built in 03A_data_analisis.ipynb by `src.sunspot_analysis.build_regions`, from the
    cubes written here. The reason is cost: the two corrections above re-read every
    per-frame FITS header, ~90 s per region, and none of that work depends on where an umbra
    boundary is drawn. Keeping the regions out of here means retuning one costs a 03A run
    rather than a full re-correction.

    The one place a mask is still needed is the magnetogram plane fit, which has to know
    which pixels are quiet sun. That uses its own pinned PLANE_QSUN_* cut rather than
    03A's thresholds — see the setup cell for why that is safe and deliberate.

    Every step is NaN-safe, because a gap slot on the uniform grid is an all-NaN frame: the
    I_qs percentile and the plane fit both guard on there being finite pixels. `present` is
    carried through so 03A can tell a gap apart from a frame where nothing was detected.
    """
    aligned    = load_aligned(region_dir, crop_to_data=crop_to_data, v_sdo_by_time=v_sdo_by_time)
    cubes      = aligned['cubes']
    timestamps = aligned['grid']
    cube_cont, cube_mag, cube_dop = cubes['cont'], cubes['mag'], cubes['dop']
    n_t, ny, nx = cube_cont.shape

    # Normalized pixel coordinates for the plane fit, so the design matrix stays well
    # conditioned regardless of box size.
    yy, xx = np.mgrid[0:ny, 0:nx]
    xn = ((xx - nx / 2) / (nx / 2)).astype(np.float64)
    yn = ((yy - ny / 2) / (ny / 2)).astype(np.float64)

    # Quiet sun for the plane fit only: everything not dark. A NaN threshold compares False
    # everywhere, so a gap frame contributes no pixels and its plane comes back NaN.
    finite = np.isfinite(cube_cont)
    i_qs = np.array([np.percentile(cube_cont[t][finite[t]], PLANE_QSUN_PERCENTILE)
                     if finite[t].any() else np.nan for t in range(n_t)])
    with np.errstate(invalid='ignore'):
        plane_qsun = finite & ~(cube_cont < (PLANE_QSUN_FRAC * i_qs)[:, None, None])

    plane_coefs = {'mag': np.full((n_t, 3), np.nan), 'dop': np.full((n_t, 3), np.nan)}
    cube_mag = cube_mag.copy()          # crop_to_common_window returns views
    cube_dop = cube_dop.copy() if RESIDUAL_PLANE_FIT else cube_dop
    for t in range(n_t):
        cube_mag[t], plane_coefs['mag'][t] = remove_quiet_sun_plane(
            cube_mag[t], plane_qsun[t], xn, yn)
        if RESIDUAL_PLANE_FIT:
            cube_dop[t], plane_coefs['dop'][t] = remove_quiet_sun_plane(
                cube_dop[t], plane_qsun[t], xn, yn)

    return dict(
        cubes={'cont': cube_cont, 'mag': cube_mag, 'dop': cube_dop},
        timestamps=timestamps, i_qs=i_qs, plane_coefs=plane_coefs,
        doppler_terms=aligned['doppler_terms'], c_means=aligned['c_means'],
        present=aligned['present'], cadence_s=aligned['cadence_s'],
        crop_offset=aligned['crop_offset'],
        plane_qsun_px=plane_qsun.reshape(n_t, -1).sum(axis=1),
    )


In [4]:
from astropy.io import fits


def region_noaa(region_dir):
    """NOAA number from a `NOAA_<number>_<date>` directory name, or None."""
    parts = region_dir.name.split('_')
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else None


def write_region(region_dir, result):
    """Write one region's three corrected cubes plus a per-frame diagnostics table.

    Four files, no masks. The regions are 03A's job and are rebuilt from these cubes every
    time it runs, so there is deliberately no mask file here to go stale against whatever
    thresholds 03A currently uses. 03A exports its own mask cube for DS9 when you want one.

    Everything goes out through src.utilities.write_cube so it comes back through read_cube
    unchanged, and so DS9 sees the same structure as the raw cubes: a 3D primary HDU
    carrying the reference frame's spatial WCS, plus a TIMESTAMPS extension.
    """
    out_dir = PROCESSED_DIR / region_dir.name
    timestamps = result['timestamps']
    present    = result['present']
    cadence_s  = result['cadence_s']
    n_gaps     = int((~(present['cont'] & present['mag'] & present['dop'])).sum())

    # Reuse the raw continuum cube's header so the corrected cubes keep the spatial WCS,
    # and DS9 puts them on the same footprint as the frames they came from.
    src_header = fits.getheader(region_dir / 'region_01_continuum_cube.fits')
    for key in ('NAXIS', 'NAXIS1', 'NAXIS2', 'NAXIS3', 'NFRAMES', 'BITPIX'):
        src_header.pop(key, None)
    src_header['CADENCE'] = (cadence_s, '[s] uniform grid spacing along axis 3')
    src_header['NGAPS']   = (n_gaps, 'frames with no data in at least one series')

    # A crop moves the origin, so the inherited CRPIX would put the cube in the wrong place
    # on the sky. FITS pixel coordinates are 1-based and the crop offset is 0-based, but
    # both refer to the same axis, so the shift is just a subtraction.
    crop_offset = result.get('crop_offset')
    if crop_offset is not None:
        row0, col0 = crop_offset
        src_header['CRPIX1'] = src_header['CRPIX1'] - col0
        src_header['CRPIX2'] = src_header['CRPIX2'] - row0
        src_header['CROPROW'] = (row0, 'first row of the crop in the original cube')
        src_header['CROPCOL'] = (col0, 'first column of the crop in the original cube')

    provenance = [
        f'02A: {len(timestamps)} frames on a uniform {cadence_s:.0f} s grid, '
        f'{n_gaps} NaN (missing) frame(s)',
        '02A: corrections only - umbra/penumbra/hot spot are built in 03A from these cubes',
    ]
    if crop_offset is not None:
        ny, nx = result['cubes']['cont'].shape[1:]
        provenance.append(f'02A: cropped to the common data window {ny}x{nx} at '
                          f'row {crop_offset[0]}, col {crop_offset[1]}; CRPIX shifted to match')

    doppler_history = (
        ['02A: dopplergram calibrated - Castellanos Duran 2021 sdo+lsf+clv+gravity']
        if CALIBRATE_DOPPLER else ['02A: dopplergram NOT calibrated'])
    if RESIDUAL_PLANE_FIT:
        doppler_history.append('02A: residual quiet-sun plane also subtracted (absolute scale lost)')

    if CORRECT_LIMB_DARKENING:
        continuum_history = [
            '02A: continuum limb-darkening corrected - Castellanos Duran & Kleint 2020 '
            f'Eq.1-2 (u={U_LAMBDA}, v={V_LAMBDA}, mu from {MU_METHOD})',
            '02A: intensity left in DN/s - Eq.3 DN->cgs factor NOT applied']
    else:
        continuum_history = ['02A: continuum NOT limb-darkening corrected']

    cont_header = src_header.copy()
    cont_header['LDCORR'] = (CORRECT_LIMB_DARKENING, 'limb darkening divided out (Duran 2020 Eq.1)')
    if CORRECT_LIMB_DARKENING:
        cont_header['LD_U']  = (U_LAMBDA, 'limb-darkening coefficient u at 6173.3 A')
        cont_header['LD_V']  = (V_LAMBDA, 'limb-darkening coefficient v at 6173.3 A')
        cont_header['LD_MU'] = (MU_METHOD, 'how cos(Theta) was obtained')

    mag_header = src_header.copy()
    mag_header['PLQSFRAC'] = (PLANE_QSUN_FRAC, 'I/I_qs above which a pixel is in the plane fit')
    mag_header['PLQSPCT']  = (PLANE_QSUN_PERCENTILE, 'percentile used as I_qs for the fit')

    written = {}
    written['continuum'] = write_cube(
        result['cubes']['cont'], out_dir / 'region_01_continuum_cube.fits',
        header=cont_header, timestamps=timestamps,
        history=provenance + continuum_history)

    written['magnetogram'] = write_cube(
        result['cubes']['mag'], out_dir / 'region_01_magnetogram_corrected_cube.fits',
        header=mag_header, timestamps=timestamps,
        history=provenance + [
            f'02A: quiet-sun plane subtracted, fitted over I >= {PLANE_QSUN_FRAC} I_qs '
            f'(I_qs = p{PLANE_QSUN_PERCENTILE})'])

    written['dopplergram'] = write_cube(
        result['cubes']['dop'], out_dir / 'region_01_dopplergram_calibrated_cube.fits',
        header=src_header, timestamps=timestamps, history=provenance + doppler_history)

    # Per-frame diagnostics for the corrections. Not analysis output — these say how big
    # each correction was, which is the only way to tell a correction that went wrong from
    # solar signal. The Doppler term means in particular used to be computed and thrown
    # away at the end of the driver cell; saving them lets 03A plot them.
    n_t = len(timestamps)
    terms = result['doppler_terms'] or {}
    c_means = result['c_means']
    if c_means is None:
        c_means = np.full(n_t, np.nan)
    grad = np.hypot(result['plane_coefs']['mag'][:, 1], result['plane_coefs']['mag'][:, 2])

    cols = [
        fits.Column(name='T_OBS', format='23A',
                    array=np.array([t.isoformat() for t in timestamps])),
        fits.Column(name='PRESENT_CONT', format='L', array=present['cont']),
        fits.Column(name='PRESENT_MAG', format='L', array=present['mag']),
        fits.Column(name='PRESENT_DOP', format='L', array=present['dop']),
        fits.Column(name='I_QS', format='E', array=result['i_qs'], unit='DN/s'),
        fits.Column(name='C_MEAN', format='E', array=c_means),
        fits.Column(name='PLANE_QSUN_PX', format='J', array=result['plane_qsun_px']),
        fits.Column(name='MAG_PLANE_GRAD', format='E', array=grad, unit='G'),
    ]
    for name, key in [('V_SDO', 'sdo'), ('V_LSF', 'lsf'),
                      ('V_CLV', 'clv'), ('V_GRAVITY', 'gravity')]:
        cols.append(fits.Column(name=name, format='E', unit='m/s',
                                array=terms.get(key, np.full(n_t, np.nan))))

    frames_path = out_dir / 'region_01_frames.fits'
    frames_path.parent.mkdir(parents=True, exist_ok=True)
    fits.BinTableHDU.from_columns(cols, name='FRAMES').writeto(frames_path, overwrite=True)
    written['frames'] = str(frames_path)

    return out_dir, written

In [9]:
results = {}

regions = regions


for region_dir in regions:
    print(region_dir.name)
    noaa = region_noaa(region_dir)
    result = process_region(region_dir, crop_to_data=crop_override.get(noaa, False))
    out_dir, written = write_region(region_dir, result)
    results[region_dir.name] = result

    n_t = len(result['timestamps'])
    span_h = (result['timestamps'][-1] - result['timestamps'][0]).total_seconds() / 3600
    print(f'  {n_t} frames over {span_h:.1f} h, box {result["cubes"]["cont"].shape[1:]} px')
    if result['c_means'] is not None:
        c = result['c_means']
        print(f'  limb darkening  <C>={np.nanmean(c):.4f}  '
              f'({np.nanmin(c):.4f} .. {np.nanmax(c):.4f}, so I is boosted by up to '
              f'{1 / np.nanmin(c):.3f}x)')
    grad = np.hypot(result['plane_coefs']['mag'][:, 1], result['plane_coefs']['mag'][:, 2])
    px = result['plane_qsun_px'][result['present']['cont']]
    print(f'  magnetogram plane  |gradient|={np.nanmean(grad):.1f} G per half-box, '
          f'fitted over {px.mean():.0f} quiet-sun px/frame')
    print(f'  -> {out_dir}')
    for name, path in written.items():
        print(f'       {name:12s} {pathlib.Path(path).name}')
print('\nRegions (umbra / penumbra / hot spot) are built in 03A_data_analisis.ipynb '
      'from these cubes.')

NOAA_11117_2010-10-27
  481 slots on a uniform 720 s grid, 2010-10-27 00:00 .. 2010-10-31 00:00
  NaN frames (no data): {'cont': 6, 'mag': 6, 'dop': 6}
      cont: slot 195 = 2010-10-28 15:00
      cont: slot 199 = 2010-10-28 15:48
      cont: slot 200 = 2010-10-28 16:00
      cont: slot 201 = 2010-10-28 16:12
      cont: slot 202 = 2010-10-28 16:24
      cont: ... and 1 more
      mag: slot 195 = 2010-10-28 15:00
      mag: slot 199 = 2010-10-28 15:48
      mag: slot 200 = 2010-10-28 16:00
      mag: slot 201 = 2010-10-28 16:12
      mag: slot 202 = 2010-10-28 16:24
      mag: ... and 1 more
      dop: slot 195 = 2010-10-28 15:00
      dop: slot 199 = 2010-10-28 15:48
      dop: slot 200 = 2010-10-28 16:00
      dop: slot 201 = 2010-10-28 16:12
      dop: slot 202 = 2010-10-28 16:24
      dop: ... and 1 more
  cropped to the common data window: cont=402x402, mag=402x402, dop=402x402 -> 402x402 (100% of the smallest input box)
  481 frames over 96.0 h, box (402, 402) px
  limb darkenin

TODO: make simplier the cropout part or just redowload the things i need

## Diagnostics — did each correction do something sane?

How big each correction was, per region. What matters here is only that nothing looks
absurd before the cubes get written; anything about the *regions* now belongs in 03A, which
is also where these same numbers can be plotted against time from `region_01_frames.fits`.

- **`I_qs`** should drift smoothly and slowly. Most of its old drift was limb darkening and
  is now divided out, so what is left is the correction's residual plus real evolution.
- **`C`** should sweep by several percent across the window as the region rotates, and be
  smallest (strongest correction) when the region is nearest the limb. A flat `C` means the
  per-frame headers aren't being read and every frame got the same correction.
- **Doppler terms** are what the old empirical plane fit used to absorb blindly. `sdo`
  swings by hundreds of m/s over a day; if it doesn't, the per-frame headers aren't being
  read.
- **Magnetogram plane** — the offset is the instrumental zero point, the gradient is what a
  scalar mean subtraction would have left behind. A gradient of tens of G per half-box is
  normal; hundreds means the fit is being pulled by spot pixels that the `PLANE_QSUN_*` cut
  failed to exclude.

In [ ]:
def summarize(region_name, result):
    n_t = len(result['timestamps'])
    present = result['present']
    n_gaps = {k: int((~v).sum()) for k, v in present.items()}
    print(f'\n{region_name}  ({n_t} slots @ {result["cadence_s"]:.0f} s, gaps {n_gaps})')

    i_qs = result['i_qs']
    print(f'  I_qs           {np.nanmin(i_qs):8.0f} .. {np.nanmax(i_qs):8.0f} DN   '
          f'({100 * np.ptp(i_qs[np.isfinite(i_qs)]) / np.nanmean(i_qs):.1f}% drift)')

    c = result['c_means']
    if c is not None:
        finite_c = c[np.isfinite(c)]
        print(f'  limb darkening C  {finite_c.min():.4f} .. {finite_c.max():.4f}   '
              f'(mean {finite_c.mean():.4f}; a flat C means the per-frame headers '
              f'are not being read)')

    terms = result['doppler_terms']
    if terms:
        print('  Doppler terms removed (m/s)     mean   peak-to-peak')
        for name, series in terms.items():
            s = series[np.isfinite(series)]
            print(f'    {name:10s} {s.mean():20.1f} {np.ptp(s):10.1f}')

    coefs = result['plane_coefs']['mag']
    if np.isfinite(coefs).any():
        grad = np.hypot(coefs[:, 1], coefs[:, 2])
        offset = coefs[:, 0]
        print(f'  magnetogram plane  |gradient| {np.nanmean(grad):6.1f} G per half-box, '
              f'offset {np.nanmean(offset):+7.1f} G')

    coefs = result['plane_coefs']['dop']
    if np.isfinite(coefs).any():
        grad = np.hypot(coefs[:, 1], coefs[:, 2])
        print(f'  dopplergram plane  |gradient| {np.nanmean(grad):6.1f} m/s per half-box '
              f'(RESIDUAL_PLANE_FIT is on — the absolute velocity scale is gone)')


for region_name, result in results.items():
    summarize(region_name, result)